In [1]:
import os
import sys
import importlib
import tensorflow as tf
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/DeepLearning/Pneumonia SoTA Classification"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Folder ditemukan :", os.path.exists(PROJECT_ROOT))
print("Isi direktori    :", os.listdir(PROJECT_ROOT))
print("GPU aktif        :", tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
Folder ditemukan : True
Isi direktori    : ['data', 'notebooks', 'src', 'saved_models', 'results']
GPU aktif        : []


In [12]:
import os
import sys
import importlib
from getpass import getpass

# Pastikan direktori proyek aktif diprioritaskan dan modul dimuat ulang
project_dir = os.getcwd()
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

importlib.invalidate_caches()
for mod_name in ["src", "src.config", "src.dataset"]:
    sys.modules.pop(mod_name, None)

from src import config, dataset, models, evaluate

# Ambil kredensial dari Colab Secrets (jika sudah disimpan) atau minta input aman saat runtime
try:
    from google.colab import userdata
    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
except Exception:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME") or input("Masukkan Kaggle Username: ")
    KAGGLE_KEY = os.environ.get("KAGGLE_KEY") or getpass("Masukkan Kaggle API Key: ")

# 1. Unduh & ekstrak dataset langsung dari Kaggle ke disk lokal VM Colab
dataset.download_kaggle_dataset(
    kaggle_username=KAGGLE_USERNAME,
    kaggle_key=KAGGLE_KEY,
)

# 2. Pembagian data seimbang (70:15:15) dengan SEED = 42
train_df, val_df, test_df = dataset.prepare_balanced_splits(force_regenerate=False)

print("Train :", len(train_df), train_df["label"].value_counts().to_dict())
print("Val   :", len(val_df), val_df["label"].value_counts().to_dict())
print("Test  :", len(test_df), test_df["label"].value_counts().to_dict())

[INFO] Dataset sudah tersedia (5856 citra: 1583 NORMAL, 4273 PNEUMONIA) di /content/pneumonia_workspace/raw
[INFO] Memuat manifest pembagian data yang sudah ada dari data/splits/...
Train : 1400 {'NORMAL': 700, 'PNEUMONIA': 700}
Val   : 300 {'PNEUMONIA': 150, 'NORMAL': 150}
Test  : 300 {'PNEUMONIA': 150, 'NORMAL': 150}


In [11]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

def set_global_seed(seed=config.SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def run_experiment(arch_name, train_df, val_df, test_df, n_reps=config.N_REPETITIONS, ep1=None, ep2=None):
    records = []
    histories = []
    best_f1 = -1.0
    best_info = {}

    for run in range(1, n_reps + 1):
        print(f"\n{'='*60}\nMODEL: {arch_name} | REPETISI {run}/{n_reps}\n{'='*60}")
        tf.keras.backend.clear_session()
        set_global_seed(config.SEED)

        train_gen, val_gen, test_gen = dataset.create_data_generators(train_df, val_df, test_df)
        model, history, duration, p1, p2 = models.train_two_phase(
            arch_name, train_gen, val_gen, epochs_p1=ep1, epochs_p2=ep2
        )
        histories.append(history)

        metrics = evaluate.evaluate_on_test_set(model, test_gen)
        records.append({
            "Model": arch_name,
            "Repetisi": run,
            "Trainable_Params_Phase1": p1,
            "Trainable_Params_Phase2": p2,
            "Training_Time_Sec": round(duration, 2),
            "Accuracy": metrics["Accuracy"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "F1_Score": metrics["F1_Score"],
            "TP": metrics["TP"],
            "TN": metrics["TN"],
            "FP": metrics["FP"],
            "FN": metrics["FN"],
        })

        print(f"[Run {run}] Acc: {metrics['Accuracy']:.4f} | Prec: {metrics['Precision']:.4f} | "
              f"Rec: {metrics['Recall']:.4f} | F1: {metrics['F1_Score']:.4f} | "
              f"FP: {metrics['FP']} | FN: {metrics['FN']} | Time: {duration:.2f}s")

        if metrics["F1_Score"] > best_f1:
            best_f1 = metrics["F1_Score"]
            save_dir = os.path.join(config.SAVED_MODELS_DIR, arch_name)
            model.save(os.path.join(save_dir, f"{arch_name}_best.keras"))
            model.save(os.path.join(save_dir, f"{arch_name}_best.h5"))
            best_info = {
                "arch_name": arch_name,
                "best_run": run,
                "accuracy": metrics["Accuracy"],
                "precision": metrics["Precision"],
                "recall": metrics["Recall"],
                "f1_score": metrics["F1_Score"],
                "cm": metrics["cm"]
            }

    return pd.DataFrame(records), histories, best_info


# --- EKSPERIMEN TUNGGAL PENUH (1 Repetisi: 10 Epoch Fase 1 + 20 Epoch Fase 2) ---
df_res, hist_res, best_res = run_experiment(
    "ResNet50", train_df, val_df, test_df, n_reps=1, ep1=10, ep2=20
)
df_mob, hist_mob, best_mob = run_experiment(
    "MobileNetV2", train_df, val_df, test_df, n_reps=1, ep1=10, ep2=20
)

all_results = pd.concat([df_res, df_mob], ignore_index=True)
all_results.to_csv(os.path.join(config.LOGS_DIR, "hasil_eksperimen_sota_1run.csv"), index=False)

summary = evaluate.summarize_results(all_results)
print("\n=== RINGKASAN EVALUASI (1 RUN PENUH) ===")
print(summary)

evaluate.plot_comparison_confusion_matrices(best_res, best_mob, all_results)
evaluate.plot_learning_curves(hist_res, "ResNet50", split_epoch=10)
evaluate.plot_learning_curves(hist_mob, "MobileNetV2", split_epoch=10)


MODEL: ResNet50 | REPETISI 1/1
Found 1400 validated image filenames belonging to 2 classes.
Found 300 validated image filenames belonging to 2 classes.
Found 300 validated image filenames belonging to 2 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10


KeyboardInterrupt: 